In [0]:
%python
from pyspark.sql.functions import col, upper, trim, levenshtein, lit, when, regexp_replace

# 1. LOAD DATA
df_bronze = spark.read.table("`prism-sentinel-stream`.prism_bronze.transactions_raw")
df_sanctions = spark.read.table("`prism-sentinel-stream`.prism_silver.sanctions_master").filter("is_current = true")

# 2. DATA GOVERNANCE & CLEANING
# Masking User ID and cleaning strings
df_cleaned = df_bronze.withColumn("masked_user_id", regexp_replace(col("user_id").cast("string"), r"(\d{3})\d+", "$1****")) \
                      .withColumn("clean_counterparty", upper(trim(col("counterparty"))))

# 3. THE EVASION HUNTER LOGIC (Fuzzy Join)
# We join transactions to sanctions where names match or are very similar (Levenstein distance < 3)
df_silver = df_cleaned.join(df_sanctions, 
    (df_cleaned.clean_counterparty == df_sanctions.entity_name) | 
    (levenshtein(df_cleaned.clean_counterparty, df_sanctions.entity_name) < 4), 
    "left"
)

# 4. FLAG THE RISK
df_final_silver = df_silver.withColumn("risk_flag", 
    when(col("entity_name").isNotNull(), lit("ALERT"))
    .otherwise(lit("CLEAN"))
)

# 5. PERSIST TO SILVER
df_final_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("`prism-sentinel-stream`.prism_silver.transactions_refined")

print("✅ Silver Layer Refined: Risk alerts generated for 20M records.")

In [0]:
%python
from pyspark.sql.functions import col, levenshtein, greatest, length, round, lit

# Load necessary tables
silver_table = "`prism-sentinel-stream`.prism_silver.transactions_refined"
df_silver = spark.read.table(silver_table)

# Calculate Similarity %: (1 - (Distance / Max Length of the two strings)) * 100
df_with_scores = df_silver.withColumn(
    "similarity_score",
    round(
        (lit(1) - (levenshtein(col("counterparty"), col("entity_name")) / 
         greatest(length(col("counterparty")), length(col("entity_name"))))) * 100, 
    2)
)

# Overwrite with the new metric
df_with_scores.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(silver_table)

print("✅ Similarity Scores calculated and updated in the Silver Layer.")